# 03 — CRS and Resolution Reconciliation

Reproject all input layers onto a common 1 km grid in EPSG:32630 (UTM Zone 30N). Document the resampling method chosen for each source and why.

**Common grid spec:**
- CRS: EPSG:32630 (WGS 84 / UTM Zone 30N)
- Resolution: 1000 m × 1000 m
- Extent: derived from AOI polygon reprojected to UTM (confirm exact pixel-aligned bounds here)

**Resampling decision table (draft — update after testing):**

| Source | Direction | Method | Rationale |
|--------|-----------|--------|-----------|
| MODIS LST 1km | Same resolution, reproject only | Bilinear | Continuous temperature field |
| MODIS NDVI 250m | Aggregate 4×4 → 1km | Mean | Fractional vegetation — mean is physically meaningful |
| SMAP SM 9km | Disaggregate 9→1km | Bilinear | No new information added; registers to grid only |
| ESA CCI SM 25km | Disaggregate 25→1km | Bilinear | Same; very coarse — consider whether to use at all |
| GPM 11km | Disaggregate 11→1km | Bilinear | Areal rainfall — bilinear preserves smoothness |
| ERA5-Land 9km | Disaggregate 9→1km | Bilinear | Continuous met fields |
| FloodWatch DEM 30m | Aggregate → 1km terrain stats | Mean elev + Std slope | Preserve terrain heterogeneity signal |

**Inputs:** `data/raw/sample/`  
**Outputs:** `data/processed/aligned/` (excluded from git)  
**Gates:** `04_temporal_alignment.ipynb`

In [ ]:
from pathlib import Path
import numpy as np
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from rasterio.crs import CRS

RAW_DIR       = Path("../../data/raw/sample")
ALIGNED_DIR   = Path("../../data/processed/aligned")
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CRS = "EPSG:32630"
TARGET_RES = 1000  # metres

## 1. Define the reference grid

Derive the pixel-aligned bounding box from the AOI polygon in UTM. All layers will be snapped to this exact grid so array indices correspond to the same ground location.

In [ ]:
# TODO: load AOI polygon (confirm source — FloodWatch boundary or manually defined)
# import geopandas as gpd
# aoi = gpd.read_file("<path to AOI shapefile>").to_crs(TARGET_CRS)
# bounds = aoi.total_bounds  # minx, miny, maxx, maxy in UTM metres
#
# Snap to TARGET_RES pixel grid:
# xmin = np.floor(bounds[0] / TARGET_RES) * TARGET_RES
# ymin = np.floor(bounds[1] / TARGET_RES) * TARGET_RES
# xmax = np.ceil(bounds[2]  / TARGET_RES) * TARGET_RES
# ymax = np.ceil(bounds[3]  / TARGET_RES) * TARGET_RES
#
# print(f"Reference grid: {xmin:.0f} {ymin:.0f} {xmax:.0f} {ymax:.0f} (UTM 30N)")
# print(f"Grid size: {(xmax-xmin)/TARGET_RES:.0f} cols × {(ymax-ymin)/TARGET_RES:.0f} rows")
print("Reference grid: stub — needs AOI polygon path")

## 2. Reproject MODIS LST

In [ ]:
# TODO:
# lst = rioxarray.open_rasterio(RAW_DIR / "modis_lst" / "<file>").squeeze()
# lst_reproj = lst.rio.reproject(
#     TARGET_CRS,
#     resolution=TARGET_RES,
#     resampling=Resampling.bilinear,
#     nodata=0,
# ).rio.clip_box(*grid_bounds_utm)
# lst_reproj.rio.to_raster(ALIGNED_DIR / "lst_aligned_<date>.tif")
print("LST reproject: stub")

## 3. Aggregate MODIS NDVI 250m → 1km

In [ ]:
# TODO: coarsen 4x4 block mean, then reproject to align exactly to reference grid
# ndvi = rioxarray.open_rasterio(RAW_DIR / "modis_ndvi" / "<file>").squeeze() * 0.0001
# ndvi_1km = ndvi.coarsen(x=4, y=4, boundary="trim").mean()
# ndvi_reproj = ndvi_1km.rio.reproject(TARGET_CRS, resolution=TARGET_RES, resampling=Resampling.bilinear)
# ndvi_reproj.rio.to_raster(ALIGNED_DIR / "ndvi_aligned_<date>.tif")
print("NDVI aggregate+reproject: stub")

## 4. Reproject SMAP, GPM, ERA5-Land (coarse → 1km)

In [ ]:
# Each follows the same bilinear reproject pattern as LST above.
# Key differences to handle:
#   SMAP: EASE-Grid 2.0 CRS (EPSG:6933) — rioxarray should handle automatically
#   ERA5-Land: lat is often stored descending — check orientation after open
#   GPM: check variable name and units (mm/hr vs mm/day depending on product version)
print("Coarse layer reproject: stubs")

## 5. Aggregate FloodWatch DEM 30m → 1km terrain stats

In [ ]:
# Compute per 1km cell:
#   - mean elevation
#   - std of slope (terrain heterogeneity proxy)
# Both become static auxiliary layers — computed once, not per time step.
#
# TODO:
# from scipy.ndimage import uniform_filter
# dem = rioxarray.open_rasterio("<floodwatch dem path>").squeeze()
# dem_reproj = dem.rio.reproject(TARGET_CRS)
# # Aggregate 33×33 block (30m*33 ≈ 1000m) → mean
# factor = int(TARGET_RES / 30)
# dem_1km = dem_reproj.coarsen(x=factor, y=factor, boundary="trim").mean()
print("DEM aggregation: stub — confirm FloodWatch DEM path")

## 6. Alignment verification

After reprojection, confirm all aligned layers share identical shape, transform, and CRS.

In [ ]:
# TODO: open all aligned .tif files and assert they share the same grid
# for f in sorted(ALIGNED_DIR.glob("*.tif")):
#     da = rioxarray.open_rasterio(f)
#     print(f"{f.name}: shape={da.shape}, crs={da.rio.crs}, res={da.rio.resolution()}")
print("Alignment verification: stub")

## 7. Resampling decisions — final record

Update the table in the markdown header once you've confirmed each layer aligns correctly. Copy the final table into `EDA_FINDINGS.md` section 3.